<a href="https://colab.research.google.com/github/Andrelhu/Simulated-PT-Patient/blob/main/claude_ver/Ana_Benchmark_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ana Agent — Multi-Model Benchmark

Runs the full Ana pipeline (Stage 1 draft + Stage 2 guardrail) across four
Ollama models using a fixed 15-question PT history-taking interview.

**Captures per call:**
- `prompt_eval_count` — input tokens
- `eval_count` — output tokens
- `eval_duration` — generation time → tokens/sec
- Wall-clock time (Python `perf_counter`)

**Models benchmarked (Q4_K_M quantized):**
| Name | Tag | Disk | VRAM |
|---|---|---|---|
| llama3.2:3b | `llama3.2:3b-instruct-q4_K_M` | 2.0 GB | 2–3 GB |
| llama3.1:8b | `llama3.1:8b-instruct-q4_K_M` | 4.9 GB | 6–7 GB |
| dolphin-llama3 | `dolphin-llama3:8b-v2.9-q4_K_M` | 4.9 GB | 6–7 GB |
| dolphin-mistral | `dolphin-mistral:7b-v2.8-q4_K_M` | 4.4 GB | 6–7 GB |

Run cells top to bottom. Results export to `/content/benchmark_results.csv`.

## 1 — Setup

In [1]:
!pip install -q anthropic ollama tenacity python-dotenv pandas tabulate

In [2]:
import os, sys, shutil, pathlib

REPO_URL = "https://github.com/Andrelhu/Simulated-PT-Patient.git"
REPO_DIR = "/content/pt-agent"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

CLAUDE_VER = os.path.join(REPO_DIR, "claude_ver")
sys.path.insert(0, CLAUDE_VER)

_spec_src = pathlib.Path(REPO_DIR) / "Documentation" / "ana_lopez_combined_ai_simulator_script.txt"
_spec_dst_dir = pathlib.Path(REPO_DIR) / "data" / "specs"
_spec_dst_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(_spec_src, _spec_dst_dir / _spec_src.name)

SPEC_FILE = str(_spec_dst_dir / _spec_src.name)
SPEC_TEXT = pathlib.Path(SPEC_FILE).read_text(encoding="utf-8").strip()

print(f"Repo:      {REPO_DIR}")
print(f"Spec file: {SPEC_FILE}")
print(f"Spec size: {len(SPEC_TEXT):,} chars")

Already up to date.
Repo:      /content/pt-agent
Spec file: /content/pt-agent/data/specs/ana_lopez_combined_ai_simulator_script.txt
Spec size: 9,986 chars


In [3]:
# Install Ollama with GPU detection support
!apt-get install -y zstd lshw pciutils 2>/dev/null | tail -1
!curl -fsSL https://ollama.com/install.sh | sh

0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import os, subprocess, time, urllib.request

os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia"

subprocess.run(["pkill", "-f", "ollama serve"], capture_output=True)
time.sleep(1)

_log_file = open("/tmp/ollama_bench.log", "w")
_proc = subprocess.Popen(["ollama", "serve"], stdout=_log_file, stderr=_log_file)

for _i in range(30):
    if _proc.poll() is not None:
        raise RuntimeError(
            f"ollama serve exited (code {_proc.returncode}).\n"
            f"{open('/tmp/ollama_bench.log').read()[-2000:]}"
        )
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=1)
        print(f"Ollama server ready ({_i + 1}s)")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(
        f"Ollama didn't start in 30s.\n{open('/tmp/ollama_bench.log').read()[-2000:]}"
    )

Ollama server ready (2s)


## 2 — Models

All tags are Q4_K_M quantized — best balance of quality and VRAM for a T4.
Add or remove entries here to change which models are benchmarked.

In [5]:
# display_name -> ollama pull tag
MODELS = {
    "llama3.2:3b":     "llama3.2:3b-instruct-q4_K_M",
    "llama3.1:8b":     "llama3.1:8b-instruct-q4_K_M",
    "dolphin-llama3":  "dolphin-llama3:8b-v2.9-q4_K_M",
    "dolphin-mistral": "dolphin-mistral:7b-v2.8-q4_K_M",
}

In [6]:
import subprocess

# Pull all models — this may take several minutes on first run.
# subprocess instead of !ollama so the loop actually iterates all four models
# (! shell escapes in Python loops only execute the last iteration in Colab).
for _name, _tag in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Pulling {_name}  ({_tag})")
    print('='*60)
    subprocess.run(["ollama", "pull", _tag], check=True)


Pulling llama3.2:3b  (llama3.2:3b-instruct-q4_K_M)

Pulling llama3.1:8b  (llama3.1:8b-instruct-q4_K_M)

Pulling dolphin-llama3  (dolphin-llama3:8b-v2.9-q4_K_M)

Pulling dolphin-mistral  (dolphin-mistral:7b-v2.8-q4_K_M)


## 3 — Question Set

15 questions spanning the full PT history-taking arc: chief complaint,
mechanism, onset, pain characteristics, symptoms, function, social history.
History is maintained across questions within each model run (realistic),
and reset between models (fair comparison).

In [7]:
QUESTIONS = [
    "What brought you in today?",
    "Can you describe how it happened?",
    "When exactly did this start?",
    "On a scale from 0 to 10, how would you rate your pain right now?",
    "Where exactly does it hurt — can you point to it?",
    "Does anything make the pain better?",
    "Does anything make it worse?",
    "Is the pain constant, or does it come and go?",
    "Has the area changed in appearance — any swelling or bruising?",
    "How is this affecting your day-to-day activities?",
    "Have you tried anything for the pain so far?",
    "Do you have any other medical conditions I should know about?",
    "Have you had any injuries to this area before?",
    "How are you getting around right now?",
    "Is there anything else you think I should know?",
]

print(f"{len(QUESTIONS)} questions loaded.")

15 questions loaded.


## 4 — Benchmark Runner

Calls the Ollama API directly (bypassing the production provider) so we can
read the raw response metadata (`eval_count`, `eval_duration`, etc.).
Uses identical system prompts to `pipeline.py`.

In [8]:
import time
import ollama as _ollama

_STAGE1_TPL = "{spec}{reminder}"

_STAGE2_TPL = """\
You are a strict compliance editor for a clinical simulation.

SIMULATOR SCRIPT (follow these rules):
{spec}

A PT student asked: \"{user_input}\"

Below is Ana's draft response. Check it against the rules above:
1. Answers only what was directly asked (no volunteered extra information)
2. Contains no medical jargon or self-diagnosis
3. Is fully in character as a patient (no \"As an AI\" or meta-text)
4. Asks at most one follow-up question

If the draft violates any rule, rewrite ONLY the offending part. \
If the draft is fully compliant, return it unchanged. \
Return ONLY the final patient message — no explanations, no labels."""

_REMINDER = (
    "\n\n[REMINDER — obey these at all times:\n"
    "- Answer ONLY what was directly asked\n"
    "- No medical diagnoses or clinical jargon\n"
    "- Stay in character as the patient\n"
    "- Ask at most ONE follow-up question per reply]"
)


def _tps(resp):
    """Tokens per second from an Ollama response object."""
    if resp and resp.eval_duration and resp.eval_count:
        return round(resp.eval_count / resp.eval_duration * 1e9, 1)
    return None


def _chat_raw(client, model_tag, system, messages, temperature):
    """Call Ollama directly; return (text, response_obj, wall_seconds)."""
    full_messages = [{"role": "system", "content": system}] + messages
    t0 = time.perf_counter()
    resp = client.chat(
        model=model_tag,
        messages=full_messages,
        options={"temperature": temperature},
    )
    wall = time.perf_counter() - t0
    return resp.message.content, resp, wall


def run_benchmark(
    model_tag,
    questions,
    spec_text,
    temperature=0.7,
    guardrail=True,
    refresh_turns=3,
):
    """Run a full benchmark session for one model; return list of record dicts."""
    client = _ollama.Client(host="http://localhost:11434")
    history = []   # [{"role": ..., "content": ...}, ...]
    records = []

    for turn, question in enumerate(questions):
        reminder = _REMINDER if (turn > 0 and turn % refresh_turns == 0) else ""

        # ── Stage 1: draft response ───────────────────────────────────────────
        sys_s1 = _STAGE1_TPL.format(spec=spec_text, reminder=reminder)
        msgs_s1 = history + [{"role": "user", "content": question}]
        draft, r1, wall1 = _chat_raw(client, model_tag, sys_s1, msgs_s1, temperature)

        # ── Stage 2: guardrail ────────────────────────────────────────────────
        r2, wall2 = None, 0.0
        if guardrail:
            sys_s2 = _STAGE2_TPL.format(spec=spec_text, user_input=question)
            final, r2, wall2 = _chat_raw(
                client, model_tag, sys_s2,
                [{"role": "user", "content": draft}],
                0.2,
            )
        else:
            final = draft

        history += [
            {"role": "user",      "content": question},
            {"role": "assistant", "content": final},
        ]

        total_tok = (
            (getattr(r1, "prompt_eval_count", 0) or 0)
            + (getattr(r1, "eval_count", 0) or 0)
            + (getattr(r2, "prompt_eval_count", 0) if r2 else 0)
            + (getattr(r2, "eval_count", 0) if r2 else 0)
        )

        records.append({
            "model":          model_tag,
            "turn":           turn + 1,
            "question":       question[:60],
            # Stage 1
            "s1_prompt_tok":  getattr(r1, "prompt_eval_count", None),
            "s1_output_tok":  getattr(r1, "eval_count", None),
            "s1_tok_per_s":   _tps(r1),
            "s1_wall_ms":     round(wall1 * 1000),
            # Stage 2
            "s2_prompt_tok":  getattr(r2, "prompt_eval_count", None) if r2 else None,
            "s2_output_tok":  getattr(r2, "eval_count", None) if r2 else None,
            "s2_tok_per_s":   _tps(r2),
            "s2_wall_ms":     round(wall2 * 1000),
            # Totals
            "total_tok":      total_tok,
            "total_wall_ms":  round((wall1 + wall2) * 1000),
        })

        print(
            f"  Q{turn+1:02d}  s1={wall1*1000:6.0f}ms "
            f"s2={wall2*1000:6.0f}ms  "
            f"tok={total_tok:5d}  "
            f"s1_tps={_tps(r1) or 'n/a'}"
        )

    return records


print("Benchmark runner ready.")

Benchmark runner ready.


## 5 — Run Benchmark

Loops over all models. Each model gets a fresh conversation history.
Expect ~2–5 min per model on a T4, depending on size.

In [9]:
GUARDRAIL    = False   # set False to skip Stage 2 and halve runtime
TEMPERATURE  = 0.7
REFRESH_TURNS = 3

all_records = []

for _name, _tag in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Model: {_name}  ({_tag})")
    print('='*60)
    _records = run_benchmark(
        model_tag=_tag,
        questions=QUESTIONS,
        spec_text=SPEC_TEXT,
        temperature=TEMPERATURE,
        guardrail=GUARDRAIL,
        refresh_turns=REFRESH_TURNS,
    )
    all_records.extend(_records)
    print(f"  → {len(_records)} turns complete")

print(f"\nBenchmark complete. Total records: {len(all_records)}")


Model: llama3.2:3b  (llama3.2:3b-instruct-q4_K_M)
  Q01  s1= 83512ms s2=     0ms  tok= 2011  s1_tps=82.1
  Q02  s1=  2730ms s2=     0ms  tok= 2099  s1_tps=81.0
  Q03  s1= 12772ms s2=     0ms  tok= 2183  s1_tps=75.3
  Q04  s1=  1096ms s2=     0ms  tok= 2307  s1_tps=80.2
  Q05  s1=  1479ms s2=     0ms  tok= 2358  s1_tps=79.3
  Q06  s1=  1293ms s2=     0ms  tok= 2447  s1_tps=79.1
  Q07  s1=  2042ms s2=     0ms  tok= 2613  s1_tps=77.8
  Q08  s1=  1700ms s2=     0ms  tok= 2668  s1_tps=77.5
  Q09  s1=  1135ms s2=     0ms  tok= 2750  s1_tps=77.4
  Q10  s1=  2286ms s2=     0ms  tok= 2924  s1_tps=76.6
  Q11  s1=  2096ms s2=     0ms  tok= 2976  s1_tps=76.9
  Q12  s1=  1069ms s2=     0ms  tok= 3054  s1_tps=76.7
  Q13  s1=  1798ms s2=     0ms  tok= 3186  s1_tps=75.1
  Q14  s1=  2175ms s2=     0ms  tok= 3246  s1_tps=74.3
  Q15  s1=  1564ms s2=     0ms  tok= 3352  s1_tps=74.3
  → 15 turns complete

Model: llama3.1:8b  (llama3.1:8b-instruct-q4_K_M)
  Q01  s1= 68279ms s2=     0ms  tok= 2000  s1_tps=3

## 6 — Results

In [10]:
import pandas as pd

df = pd.DataFrame(all_records)

# Save raw per-turn data
df.to_csv("/content/benchmark_results.csv", index=False)
print("Saved: /content/benchmark_results.csv")

df

Saved: /content/benchmark_results.csv


,model,turn,question,s1_prompt_tok,s1_output_tok,s1_tok_per_s,s1_wall_ms,s2_prompt_tok,s2_output_tok,s2_tok_per_s,s2_wall_ms,total_tok,total_wall_ms
0,llama3.2:3b-instruct-q4_K_M,1,What brought you in today?,1936,75,82.1,83512,None,None,None,0,2011,83512
1,llama3.2:3b-instruct-q4_K_M,2,Can you describe how it happened?,2027,72,81.0,2730,None,None,None,0,2099,2730
2,llama3.2:3b-instruct-q4_K_M,3,When exactly did this start?,2114,69,75.3,12772,None,None,None,0,2183,12772
3,llama3.2:3b-instruct-q4_K_M,4,"On a scale from 0 to 10, how would you rate yo...",2257,50,80.2,1096,None,None,None,0,2307,1096
4,llama3.2:3b-instruct-q4_K_M,5,Where exactly does it hurt — can you point to it?,2282,76,79.3,1479,None,None,None,0,2358,1479
5,llama3.2:3b-instruct-q4_K_M,6,Does anything make the pain better?,2374,73,79.1,1293,None,None,None,0,2447,1293
6,llama3.2:3b-instruct-q4_K_M,7,Does anything make it worse?,2508,105,77.8,2042,None,None,None,0,2613,2042
7,llama3.2:3b-instruct-q4_K_M,8,"Is the pain constant, or does it come and go?",2588,80,77.5,1700,None,None,None,0,2668,1700
8,llama3.2:3b-instruct-q4_K_M,9,Has the area changed in appearance — any swell...,2690,60,77.4,1135,None,None,None,0,2750,1135
9,llama3.2:3b-instruct-q4_K_M,10,How is this affecting your day-to-day activities?,2815,109,76.6,2286,None,None,None,0,2924,2286


In [12]:
import pandas as pd

# Per-model summary
summary = df.groupby("model").agg(
    avg_total_tok    =("total_tok",     "mean"),
    avg_s1_wall_ms   =("s1_wall_ms",    "mean"),
    avg_s2_wall_ms   =("s2_wall_ms",    "mean"),
    avg_total_wall_ms=("total_wall_ms", "mean"),
    avg_s1_tps       =("s1_tok_per_s",  "mean"),
    avg_s2_tps       =("s2_tok_per_s",  "mean"),
).round(1)

# Overall tok/s = total output tokens / total wall time (seconds)
def _overall_tps(g):
    out_tok = g["s1_output_tok"].sum() + g["s2_output_tok"].fillna(0).sum()
    wall_s  = g["total_wall_ms"].sum() / 1000.0
    return round(out_tok / wall_s, 1) if wall_s > 0 else None

summary["overall_tok_per_s"] = df.groupby("model").apply(_overall_tps)

print(summary.to_markdown())
summary

| model                          |   avg_total_tok |   avg_s1_wall_ms |   avg_s2_wall_ms |   avg_total_wall_ms |   avg_s1_tps |   avg_s2_tps |   overall_tok_per_s |
|:-------------------------------|----------------:|-----------------:|-----------------:|--------------------:|-------------:|-------------:|--------------------:|
| dolphin-llama3:8b-v2.9-q4_K_M  |          2628.7 |           4056.9 |                0 |              4056.9 |         31.7 |          nan |                11.9 |
| dolphin-mistral:7b-v2.8-q4_K_M |          3288   |           6065.5 |                0 |              6065.5 |         28.9 |          nan |                17.6 |
| llama3.1:8b-instruct-q4_K_M    |          2600.1 |           7054.2 |                0 |              7054.2 |         34.6 |          nan |                 9   |
| llama3.2:3b-instruct-q4_K_M    |          2678.3 |           7916.5 |                0 |              7916.5 |         77.6 |          nan |                 9.7 |


/tmp/ipykernel_5334/3874570121.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out_tok = g["s1_output_tok"].sum() + g["s2_output_tok"].fillna(0).sum()
/tmp/ipykernel_5334/3874570121.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary["overall_tok_per_s"] = df.groupby("model").apply(_overall_tps)


,avg_total_tok,avg_s1_wall_ms,avg_s2_wall_ms,avg_total_wall_ms,avg_s1_tps,avg_s2_tps,overall_tok_per_s
model,,,,,,,
dolphin-llama3:8b-v2.9-q4_K_M,2628.7,4056.9,0.0,4056.9,31.7,NaN,11.9
dolphin-mistral:7b-v2.8-q4_K_M,3288.0,6065.5,0.0,6065.5,28.9,NaN,17.6
llama3.1:8b-instruct-q4_K_M,2600.1,7054.2,0.0,7054.2,34.6,NaN,9.0
llama3.2:3b-instruct-q4_K_M,2678.3,7916.5,0.0,7916.5,77.6,NaN,9.7
